In [1]:
!pip -q install pandas nltk spacy scikit-learn scipy joblib

import os, json, re, string
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import spacy

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz
import joblib

In [13]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

!python -m spacy download en_core_web_sm

nlp = spacy.load("en_core_web_sm")
stop_words = set(stopwords.words("english"))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 100.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [14]:
RAW_PATH = "youtube_comments_raw.jsonl"   # change if needed
OUT_DIR = "phase2_youtube_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

print("RAW_PATH:", RAW_PATH)
print("OUT_DIR:", OUT_DIR)

RAW_PATH: youtube_comments_raw.jsonl
OUT_DIR: phase2_youtube_outputs


In [15]:
rows = []
with open(RAW_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

df_raw = pd.DataFrame(rows)

print("Loaded rows:", len(df_raw))
print("Columns:", list(df_raw.columns))
df_raw.head(5)

Loaded rows: 3896
Columns: ['kind', 'commentId', 'videoId', 'publishedAt', 'updatedAt', 'textDisplay', 'likeCount']


,kind,commentId,videoId,publishedAt,updatedAt,textDisplay,likeCount
0,youtube#comment,Ugz3TjxB2c5U0i01kht4AaABAg,Z2UqlSo3G-A,2026-02-11T18:29:35Z,2026-02-11T18:29:35Z,Crazy.,0
1,youtube#comment,Ugz3M8msHzpXu3-VZsJ4AaABAg,Z2UqlSo3G-A,2026-02-11T18:27:52Z,2026-02-11T18:27:52Z,They aren't going to let us have this vaccine....,0
2,youtube#comment,UgwQPhsmvJ6RkrehQ1V4AaABAg,Z2UqlSo3G-A,2026-02-11T18:27:09Z,2026-02-11T18:27:09Z,Well...the rich will get vaccinated abroad if ...,0
3,youtube#comment,UgxjwrOXJBWY8E1K-Fp4AaABAg,Z2UqlSo3G-A,2026-02-11T18:23:59Z,2026-02-11T18:23:59Z,I think I may have to get my vaccines in Canad...,0
4,youtube#comment,UgySblFtCIhn_kccex14AaABAg,Z2UqlSo3G-A,2026-02-11T18:21:21Z,2026-02-11T18:21:21Z,"So, Americans over 50 may have to get vaccinat...",0


In [16]:
print("Missing values per column:")
print(df_raw.isna().sum().sort_values(ascending=False))

if "commentId" in df_raw.columns:
    print("\nUnique commentId:", df_raw["commentId"].nunique())

Missing values per column:
kind           0
commentId      0
videoId        0
publishedAt    0
updatedAt      0
textDisplay    0
likeCount      0
dtype: int64

Unique commentId: 3896


In [17]:
url_pattern = re.compile(r"https?://\S+|www\.\S+")

def clean_text_safe(s):
    if s is None:
        return ""
    s = str(s)
    s = s.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    s = url_pattern.sub("", s)          # remove URLs
    s = re.sub(r"\s+", " ", s).strip()  # normalize whitespace
    return s

In [18]:
df = df_raw.copy()

# Drop duplicates by unique commentId
if "commentId" in df.columns:
    before = len(df)
    df = df.drop_duplicates(subset=["commentId"])
    print("Dropped duplicate commentId:", before - len(df))

# Backup: drop exact duplicates
before = len(df)
df = df.drop_duplicates()
print("Dropped exact duplicate rows:", before - len(df))

# Clean text
df["text_clean"] = df["textDisplay"].apply(clean_text_safe)

# Remove empty after cleaning
before = len(df)
df = df[df["text_clean"].str.len() > 0]
print("Removed empty text rows:", before - len(df))

# Parse timestamps
df["publishedAt_dt"] = pd.to_datetime(df.get("publishedAt"), errors="coerce", utc=True)
df["updatedAt_dt"] = pd.to_datetime(df.get("updatedAt"), errors="coerce", utc=True)

print("Rows after cleaning:", len(df))
df[["commentId", "videoId", "publishedAt_dt", "textDisplay", "text_clean"]].head(5)

Dropped duplicate commentId: 0
Dropped exact duplicate rows: 0
Removed empty text rows: 22
Rows after cleaning: 3874


,commentId,videoId,publishedAt_dt,textDisplay,text_clean
0,Ugz3TjxB2c5U0i01kht4AaABAg,Z2UqlSo3G-A,2026-02-11 18:29:35+00:00,Crazy.,Crazy.
1,Ugz3M8msHzpXu3-VZsJ4AaABAg,Z2UqlSo3G-A,2026-02-11 18:27:52+00:00,They aren't going to let us have this vaccine....,They aren't going to let us have this vaccine....
2,UgwQPhsmvJ6RkrehQ1V4AaABAg,Z2UqlSo3G-A,2026-02-11 18:27:09+00:00,Well...the rich will get vaccinated abroad if ...,Well...the rich will get vaccinated abroad if ...
3,UgxjwrOXJBWY8E1K-Fp4AaABAg,Z2UqlSo3G-A,2026-02-11 18:23:59+00:00,I think I may have to get my vaccines in Canad...,I think I may have to get my vaccines in Canad...
4,UgySblFtCIhn_kccex14AaABAg,Z2UqlSo3G-A,2026-02-11 18:21:21+00:00,"So, Americans over 50 may have to get vaccinat...","So, Americans over 50 may have to get vaccinat..."


In [19]:
def preprocess_advanced(text):
    if not isinstance(text, str):
        return ""

    # lowercase
    text = text.lower()

    # remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # tokenize
    tokens = word_tokenize(text)

    # remove stopwords + short tokens + non-alphabetic
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words and len(t) > 2]

    if not tokens:
        return ""

    # lemmatize (spaCy)
    doc = nlp(" ".join(tokens))
    lemmas = [tok.lemma_ for tok in doc if tok.lemma_ and tok.lemma_ != "-PRON-"]

    return " ".join(lemmas)

In [20]:
df["text_processed"] = df["text_clean"].apply(preprocess_advanced)

# Remove rows that become empty after preprocessing
before = len(df)
df = df[df["text_processed"].str.len() > 0]
print("Removed empty after advanced preprocessing:", before - len(df))

df[["text_clean", "text_processed"]].head(5)

Removed empty after advanced preprocessing: 93


,text_clean,text_processed
0,Crazy.,crazy
1,They aren't going to let us have this vaccine....,be not go let vaccine vaccine ffs mrna vaccine...
2,Well...the rich will get vaccinated abroad if ...,wellthe rich get vaccinate abroad good
3,I think I may have to get my vaccines in Canad...,think may get vaccine move
4,"So, Americans over 50 may have to get vaccinat...",americans may get vaccinate covid outside fda ...


In [28]:
# ---------------- Vectorization (TF-IDF) ----------------
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz
import joblib, os
import pandas as pd
import numpy as np

# 1) Clean corpus (handle NaN + remove empty)
corpus = df["text_processed"].fillna("").astype(str)
mask = corpus.str.strip().str.len() > 0
df_vec = df.loc[mask].copy()
corpus = corpus.loc[mask]

# 2) TF-IDF on processed text
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

X = vectorizer.fit_transform(corpus)
print("TF-IDF shape:", X.shape)

# 3) Save artifacts
tfidf_dir = os.path.join(OUT_DIR, "tfidf")
os.makedirs(tfidf_dir, exist_ok=True)

X_path = os.path.join(tfidf_dir, "youtube_tfidf_matrix.npz")
save_npz(X_path, X)
print("Saved TF-IDF matrix:", X_path)

vec_path = os.path.join(tfidf_dir, "youtube_tfidf_vectorizer.joblib")
joblib.dump(vectorizer, vec_path)
print("Saved vectorizer:", vec_path)

# Save features
feat_path = os.path.join(tfidf_dir, "youtube_tfidf_features.csv")
pd.DataFrame({"feature": vectorizer.get_feature_names_out()}).to_csv(feat_path, index=False)
print("Saved features:", feat_path)

# Save row mapping
map_path = os.path.join(tfidf_dir, "youtube_tfidf_row_mapping.csv")
df_vec[["commentId"]].reset_index(drop=True).to_csv(map_path, index=False)
print("Saved row mapping:", map_path)

# 4) Top 10 terms by total TF-IDF weight
term_scores = np.asarray(X.sum(axis=0)).ravel()
top_idx = term_scores.argsort()[::-1][:10]

top_terms = vectorizer.get_feature_names_out()[top_idx]
top_scores = term_scores[top_idx]

print("\nTop 10 terms (total TF-IDF):")
for t, s in zip(top_terms, top_scores):
    print(t, round(float(s), 6))

TF-IDF shape: (3781, 5000)
Saved TF-IDF matrix: phase2_youtube_outputs/tfidf/youtube_tfidf_matrix.npz
Saved vectorizer: phase2_youtube_outputs/tfidf/youtube_tfidf_vectorizer.joblib
Saved features: phase2_youtube_outputs/tfidf/youtube_tfidf_features.csv
Saved row mapping: phase2_youtube_outputs/tfidf/youtube_tfidf_row_mapping.csv

Top 10 terms (total TF-IDF):
flu 187.239012
get 169.413132
shot 135.970854
flu shot 109.686512
vaccine 107.36676
get flu 87.347574
never 86.30396
not 81.817449
year 74.366837
thank 70.309967


In [29]:
# ---------------- Save Final Cleaned Dataset ----------------

clean_path = os.path.join(OUT_DIR, "youtube_phase2_cleaned.csv")

df_vec.to_csv(clean_path, index=False)

print("Saved cleaned structured dataset:", clean_path)
print("Final dataset shape:", df_vec.shape)

Saved cleaned structured dataset: phase2_youtube_outputs/youtube_phase2_cleaned.csv
Final dataset shape: (3781, 11)


In [30]:
# ---------------- Add Basic Text Statistics ----------------

df_vec["char_count"] = df_vec["text_clean"].str.len()
df_vec["word_count"] = df_vec["text_processed"].str.split().apply(len)

print(df_vec[["char_count", "word_count"]].describe())

        char_count   word_count
count  3781.000000  3781.000000
mean    146.909548    14.156837
std     206.564924    19.094377
min       3.000000     1.000000
25%      40.000000     4.000000
50%      84.000000     8.000000
75%     173.000000    17.000000
max    3162.000000   304.000000


Preprocessing Validation (Pre-EDA Preview)

This step provides a quick frequency check of the processed corpus to verify that tokenization, stopword removal, and lemmatization were applied correctly.

This is not the formal exploratory data analysis phase. Full EDA, including visualization and interpretation, will be conducted separately.

In [31]:
# ---------------- Top Frequent Terms (Raw Count) ----------------

from collections import Counter

all_tokens = " ".join(df_vec["text_processed"]).split()
word_freq = Counter(all_tokens)

top_20 = word_freq.most_common(20)

print("Top 20 Most Frequent Words:")
for word, count in top_20:
    print(word, count)

Top 20 Most Frequent Words:
flu 2272
get 1985
shot 1367
vaccine 962
not 743
year 739
take 625
never 615
do 484
sick 441
covid 389
people 384
one 339
like 331
time 302
shoot 298
make 297
doctor 260
every 257
know 256
